# IND320 Compulsory Assignment 4

Elias Vinsrygg

In this notebook, we collect data for NO3 and test how well River can predict reservoir filling for the coming months.


## 1. Project links

- Public GitHub repository: [IND320-EliasVinsrygg](https://github.com/EliasVinsrygg/IND320-EliasVinsrygg)
- Published Streamlit app: [IND320 Energy and Weather](https://ind320-eliasvinsrygg.streamlit.app/)
- CA4 screencast: *Link to be added.*

## 2. Work Log

- We downloaded reservoir, currency, weather and physical-flow data for 2016-2025.
- We checked time references, prepared weekly data and investigated missing flow observations.
- We compared River models with different lags, seasonal settings, data sources and history lengths.
- We compared weekly and four-week inputs, evaluated the selected setup on 2025, and reran the experiments with a 0-100% output limit.

- We added a NO3 forecast page with selectable model settings, sources, training dates and horizon.
- We selected the Quantile-loss interval bonus and checked its coverage on the existing validation periods.

*The remaining app changes, overall navigation/design review and screencast are still pending.*


## 3. AI Usage

I used Codex to help prepare and revise the code, explain the methods, check results and edit the notebook text. Codex also ran local checks on the downloaded data. I ran the notebook cells and discussed the results with Codex while choosing the next steps.


## 4. NB data

We first make a function that downloads EUR/NOK and USD/NOK from Norges Bank. We can choose the dates, or use the default period of one year ending today. The next cell calls the function and shows the first five rows.


In [4]:
# Reading the API's CSV response directly from memory.
from io import StringIO

import pandas as pd
import requests


def download_exchange_rates(start_date=None, end_date=None):
    """Return NB's daily USD/NOK and EUR/NOK data for the selected dates."""
    # Using today's date and one calendar year of history when dates are omitted.
    end = pd.Timestamp.today().normalize() if end_date is None else pd.Timestamp(end_date)
    start = end - pd.DateOffset(years=1) if start_date is None else pd.Timestamp(start_date)
    if pd.isna(start) or pd.isna(end) or start > end:
        raise ValueError("Dates must be valid, with start_date on or before end_date.")

    # Passing the selected dates as query parameters to the daily exchange-rate endpoint.
    url = "https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP"
    params = {
        "startPeriod": start.strftime("%Y-%m-%d"),
        "endPeriod": end.strftime("%Y-%m-%d"),
        "format": "csv",
        "locale": "en",
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    # Parsing the semicolon-separated CSV while retaining the API's column names.
    return pd.read_csv(StringIO(response.text), sep=";")


In [2]:
# Downloading the default one-year period and previewing the returned table.
exchange_rates = download_exchange_rates()
exchange_rates.head()


,FREQ,Frequency,BASE_CUR,Base Currency,QUOTE_CUR,Quote Currency,TENOR,Tenor,DECIMALS,CALCULATED,UNIT_MULT,Unit Multiplier,COLLECTION,Collection Indicator,TIME_PERIOD,OBS_VALUE
0,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-24,10.2170
1,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-25,10.2571
2,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-26,10.2330
3,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-27,10.1873
4,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-11-28,10.1716


## 5. Timezone investigation

Before combining the sources, we need to check what their dates and times mean. A weekly observation date is different from a timestamp, and the time of publication can be later than the observation.


| Source | Time reference | How we handle it |
| --- | --- | --- |
| [Magasinstatistikk](https://webfileservice.nve.no/API/PublishedFiles/Download/64edbe40-b965-4c90-a114-88162ffff454/201903383/3435075) | Weekly date, measured Sunday 24:00 or the nearest earlier available reading; no UTC offset is supplied. | We keep the weekly label. The exact timezone is not confirmed. |
| [ENTSO-E](https://transparencyplatform.zendesk.com/hc/en-us/articles/12786431986964-Response-Time-Zone) | The API returns UTC timestamps. | We keep UTC and convert to Norwegian calendar weeks when aggregating. |
| [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api) | Our responses use GMT/UTC. Precipitation covers the preceding hour; temperature is instantaneous. | We account for the covered interval when assigning precipitation to a week. |
| [Norges Bank](https://www.norges-bank.no/en/news-events/news/Press-releases/2015/Press-release-07122015/) | Daily observation dates; metadata say 14:15 CET, with publication around 16:00 CET. | We keep the dates rather than treating them as midnight UTC. |

In the example below, we see that Oslo is one hour ahead of UTC in winter and two hours ahead in summer. We therefore use UTC for timestamps and `Europe/Oslo` for calendar boundaries. The model tests later use historical observation weeks; they do not simulate publication delays.



In [5]:
import pandas as pd

# Converting winter and summer UTC timestamps while preserving the same instants.
utc_times = pd.to_datetime(["2024-01-15 12:00", "2024-07-15 12:00"], utc=True)
timezone_example = pd.DataFrame({"UTC": utc_times, "Europe/Oslo": utc_times.tz_convert("Europe/Oslo")})
timezone_example


,UTC,Europe/Oslo
0,2024-01-15 12:00:00+00:00,2024-01-15 13:00:00+01:00
1,2024-07-15 12:00:00+00:00,2024-07-15 14:00:00+02:00


## 6. Independent data preparation, 2016-2025

We prepare the four sources separately before joining them. The raw responses are saved in `data/ca4/2016_2025/`, so completed downloads can be reused without MongoDB or the app loaders. We reuse the fixed 100 NO3 weather points and the existing private ENTSO-E token.

We use complete Monday-Sunday weeks, labelled by Sunday. Filling is the final observation, temperature and currency are means, and precipitation and flow energy are sums. We leave incomplete flow totals missing.



### Getting ready for data downloading

Here we import the packages, locate the data folder and define the common weeks. The download helper saves completed responses and reuses them on later runs. This setup cell does not start any downloads.


In [6]:
# Preparing shared dependencies for the four independent data sources.
from pathlib import Path
import json
import time
import tomllib
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests

# Locating shared configuration without changing the notebook's working folder.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "Compulsory Assignments/assets/weather_locations.json").is_file()
)
DATA_FOLDER = PROJECT_ROOT / "Compulsory Assignments/data/ca4/2016_2025"
START_DATE, END_DATE = pd.Timestamp("2016-01-01"), pd.Timestamp("2026-01-01")
LOCAL_ZONE = "Europe/Oslo"

# Keeping only full local calendar weeks, with Sunday as their date label.
weeks = pd.date_range(START_DATE, END_DATE, freq="W-SUN", inclusive="left", name="date")
weeks = weeks[(weeks - pd.Timedelta(days=6) >= START_DATE)
              & (weeks + pd.Timedelta(days=1) <= END_DATE)]
week_starts = (weeks - pd.Timedelta(days=6)).tz_localize(LOCAL_ZONE).tz_convert("UTC")
week_ends = (weeks + pd.Timedelta(days=1)).tz_localize(LOCAL_ZONE).tz_convert("UTC")
expected_week_hours = pd.Series((week_ends - week_starts) / pd.Timedelta(hours=1), index=weeks)


def download_text(url, params, filename):
    """Download one response, or reuse a CA4 checkpoint; never retry HTTP errors."""
    path = DATA_FOLDER / filename
    if path.is_file():
        return path.read_text(encoding="utf-8")
    # Hiding request URLs because ENTSO-E query parameters contain the API token.
    try:
        response = requests.get(url, params=params, timeout=(15, 180))
    except requests.RequestException:
        raise RuntimeError("API connection failed; completed CA4 files are preserved.") from None
    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code}; no automatic retry. "
                           f"Retry-After: {response.headers.get('Retry-After', 'not supplied')}")
    response.encoding = "utf-8"
    # Saving completed responses atomically so an interruption leaves no partial checkpoint.
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(response.text, encoding="utf-8")
    temporary.replace(path)
    return response.text


### 6.1 Magasinstatistikk: download

We download the full [NVE reservoir history](https://api.nve.no/doc/magasinstatistikk/), or read the saved response. We select NO3 in the next cell.


In [7]:
# Downloading NVE's JSON directly; this endpoint returns the full reservoir history.
nve_raw = json.loads(download_text(
    "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData",
    {}, "reservoirs.json",
))
print(f"NVE: {len(nve_raw)} source records.")


NVE: 14976 source records.


### 6.1 Magasinstatistikk: processing

We select NO3 as `EL/3` and convert filling ratios to percentages and weekly changes to percentage points. Stored energy and capacity stay in TWh. We keep the extra columns for analysis, but do not use same-week stored energy or filling change to predict the filling they already describe.


In [8]:
# Selecting NO3 (EL/3), retaining source fields for analysis and later feature selection.
reservoir_columns = {
    "dato_Id": "date",
    "omrType": "area_type",
    "omrnr": "area_number",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "filling_percent",
    "fylling_TWh": "stored_energy_twh",
    "kapasitet_TWh": "capacity_twh",
    "fyllingsgrad_forrige_uke": "previous_week_filling_percent",
    "endring_fyllingsgrad": "weekly_change_pp",
}
reservoir = pd.DataFrame(nve_raw)
reservoir = reservoir.loc[
    (reservoir["omrType"] == "EL") & (reservoir["omrnr"] == 3),
    list(reservoir_columns),
].rename(columns=reservoir_columns).copy()

# Keeping weekly date labels without inventing UTC instants, and validating numeric fields.
reservoir["date"] = pd.to_datetime(reservoir["date"], errors="raise")
numeric_columns = [column for column in reservoir.columns if column not in {"date", "area_type"}]
reservoir[numeric_columns] = reservoir[numeric_columns].apply(pd.to_numeric, errors="raise")
if (reservoir["date"].isna().any() or reservoir["date"].duplicated().any()
        or not np.isfinite(reservoir[numeric_columns].to_numpy(dtype=float)).all()):
    raise ValueError("Reservoir dates must be unique and present, and numeric values must be finite.")

# Converting source ratios to percent and changes to percentage points; retaining TWh units.
percent_columns = ["filling_percent", "previous_week_filling_percent", "weekly_change_pp"]
reservoir[percent_columns] = reservoir[percent_columns] * 100
if (not reservoir[["filling_percent", "previous_week_filling_percent"]].apply(
        lambda column: column.between(0, 100)).all().all()
        or not reservoir["weekly_change_pp"].between(-100, 100).all()
        or not reservoir["capacity_twh"].gt(0).all()
        or not reservoir["stored_energy_twh"].ge(0).all()):
    raise ValueError("Unexpected filling percentages, weekly changes, stored energy or capacity.")

# Verifying the source ISO year/week against the date, including weeks around New Year.
iso_calendar = reservoir["date"].dt.isocalendar()
if (not reservoir["iso_year"].eq(iso_calendar["year"]).all()
        or not reservoir["iso_week"].eq(iso_calendar["week"]).all()
        or not reservoir["date"].dt.dayofweek.eq(6).all()):
    raise ValueError("Reservoir ISO fields must match their Sunday observation dates.")

# Checking every target week without interpolating gaps or discarding ISO week 53.
reservoir_weekly = reservoir.set_index("date").sort_index().reindex(weeks)
if reservoir_weekly.isna().any().any():
    raise ValueError("Missing NO3 reservoir weeks or fields; inspect the source before modelling.")
reservoir_weekly.head()


,area_type,area_number,iso_year,iso_week,filling_percent,stored_energy_twh,capacity_twh,previous_week_filling_percent,weekly_change_pp
date,,,,,,,,,
2016-01-10,EL,3,2016,1,68.995315,6.155025,8.920932,72.939220,-3.943902
2016-01-17,EL,3,2016,2,64.587426,5.761800,8.920932,68.995315,-4.407889
2016-01-24,EL,3,2016,3,59.888100,5.342576,8.920932,64.587426,-4.699326
2016-01-31,EL,3,2016,4,57.638526,5.141893,8.920932,59.888100,-2.249575
2016-02-07,EL,3,2016,5,54.853940,4.893482,8.920932,57.638526,-2.784586


### 6.2 Norges Bank: download

We now request daily EUR/NOK and USD/NOK for the full 2016-2025 period from the [NB currency API](https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP?startPeriod=2016-01-01&endPeriod=2025-12-31&format=csv&locale=en). The saved response is reused if it already exists.


In [9]:
from io import StringIO

# Downloading daily USD/NOK and EUR/NOK for the entire independent CA4 period.
currency_raw = pd.read_csv(StringIO(download_text(
    "https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP",
    {"startPeriod": "2016-01-01", "endPeriod": "2025-12-31", "format": "csv", "locale": "en"},
    "currency.csv",
)), sep=";")
print(f"NB: {len(currency_raw)} source records.")


NB: 5032 source records.


### 6.2 Norges Bank: processing

We check the currency units and calculate the average available rate for each week. We do not invent observations for weekends or holidays. The two resulting columns can now be matched to the reservoir weeks.


In [10]:
# Keeping observation dates as dates; the API provides no per-row UTC timestamp.
currency = currency_raw[["TIME_PERIOD", "BASE_CUR", "QUOTE_CUR", "UNIT_MULT", "OBS_VALUE"]].copy()
currency["date"] = pd.to_datetime(currency["TIME_PERIOD"], errors="raise")
currency["rate"] = pd.to_numeric(currency["OBS_VALUE"], errors="raise")
if (not currency["QUOTE_CUR"].eq("NOK").all()
        or not pd.to_numeric(currency["UNIT_MULT"], errors="raise").eq(0).all()
        or set(currency["BASE_CUR"]) != {"USD", "EUR"}
        or not np.isfinite(currency["rate"]).all() or not currency["rate"].gt(0).all()
        or currency.duplicated(["date", "BASE_CUR"]).any()):
    raise ValueError("Unexpected currency units, values or duplicate observations.")

# Averaging available business-day observations, without inventing weekend/holiday rates.
currency_daily = currency.pivot(index="date", columns="BASE_CUR", values="rate").sort_index()
currency_weekly = currency_daily.resample("W-SUN").mean().reindex(weeks).rename(
    columns={"USD": "usd_nok", "EUR": "eur_nok"})
currency_weekly.columns.name = None
if currency_weekly.isna().any().any():
    raise ValueError("A target week has no exchange-rate observations.")
currency_weekly.head()


,eur_nok,usd_nok
date,,
2016-01-10,9.64730,8.91384
2016-01-17,9.61804,8.84878
2016-01-24,9.62910,8.85532
2016-01-31,9.46522,8.70568
2016-02-07,9.51122,8.62502


### 6.3 ENTSO-E: download

We download physical flows between NO3 and SE2, NO5, NO1 and NO4, using the [ENTSO-E area codes](https://transparencyplatform.zendesk.com/hc/en-us/articles/15885757676308-Area-List-with-Energy-Identification-Code-EIC). Import means flow into NO3, and export means flow out of NO3.

This gives eight series and 960 monthly requests for 2016-2025. Completed XML files are saved under `entsoe/NO3/` and skipped on reruns. The cell stops on HTTP errors, so a failed run can be restarted later. [API parameters](https://transparencyplatform.zendesk.com/hc/en-us/articles/15696716612372-Request-Parameters).


In [16]:
# Reading the existing token privately; no Cassandra, Spark or MongoDB is involved.
with (PROJECT_ROOT / ".streamlit/secrets.toml").open("rb") as secrets_file:
    entsoe_token = tomllib.load(secrets_file)["entsoe"]["security_token"]

# Mapping the selected bidding zones to ENTSO-E EIC codes, rather than country codes.
no3_code = "10YNO-3--------J"
neighbour_codes = {
    "SE2": "10Y1001A1001A45N",
    "NO5": "10Y1001A1001A48H",
    "NO1": "10YNO-1--------2",
    "NO4": "10YNO-4--------9",
}
month_edges = pd.date_range(START_DATE, END_DATE, freq="MS", tz="UTC")
entsoe_files = []
for neighbour, neighbour_code in neighbour_codes.items():
    # Defining export as NO3 -> neighbour and import as neighbour -> NO3.
    for direction, origin, destination in (("export", no3_code, neighbour_code),
                                           ("import", neighbour_code, no3_code)):
        for start, end in zip(month_edges[:-1], month_edges[1:]):
            # Keeping these checkpoints separate from any national-flow downloads.
            filename = f"entsoe/NO3/{neighbour}_{direction}_{start:%Y_%m}.xml"
            if not (DATA_FOLDER / filename).is_file():
                time.sleep(0.3)
            download_text("https://web-api.tp.entsoe.eu/api",
                          {"securityToken": entsoe_token, "documentType": "A11",
                           "out_Domain": origin, "in_Domain": destination,
                           "periodStart": start.strftime("%Y%m%d%H%M"),
                           "periodEnd": end.strftime("%Y%m%d%H%M")}, filename)
            entsoe_files.append((filename, neighbour, direction, origin, destination, start, end))
        print(f"ENTSO-E: NO3 / {neighbour} {direction}, monthly responses saved.", flush=True)
del entsoe_token


ENTSO-E: NO3 / SE2 export, monthly responses saved.
ENTSO-E: NO3 / SE2 import, monthly responses saved.
ENTSO-E: NO3 / NO5 export, monthly responses saved.
ENTSO-E: NO3 / NO5 import, monthly responses saved.
ENTSO-E: NO3 / NO1 export, monthly responses saved.
ENTSO-E: NO3 / NO1 import, monthly responses saved.
ENTSO-E: NO3 / NO4 export, monthly responses saved.
ENTSO-E: NO3 / NO4 import, monthly responses saved.


### 6.3 ENTSO-E: processing

We read the [A01/A03 intervals](https://transparencyplatform.zendesk.com/hc/en-us/articles/30262342482961-CurveType-A01-vs-CurveType-A03) and calculate energy as MW multiplied by hours. A01 covers one time step, while A03 lasts until the next change or period end. We split intervals at Norwegian week boundaries, using the [UTC timestamps](https://transparencyplatform.zendesk.com/hc/en-us/articles/12786431986964-Response-Time-Zone).

We only accept a weekly MWh total when the whole week is covered. Missing data stays `NaN`, while a published zero stays zero. For example, `no3_se2_import_mwh` is energy flowing from SE2 into NO3.

Course: [Time formats](../../Curriculum/D2Dbook/3_Data_quality/3_Preprocessing/3_Time_formats.ipynb), "ISO standard" and "Time conversion"; [REST](../../Curriculum/D2Dbook/2_Data_sources/3_APIs/2_REST.ipynb), "Retrieving data (GET)".


In [17]:
def parse_flow_response(text, origin, destination, request_start, request_end):
    """Read published MW intervals, clipping them to one non-overlapping request."""
    root = ET.fromstring(text)
    if root.tag.endswith("Acknowledgement_MarketDocument"):
        reason = root.findtext(".//{*}Reason/{*}text") or "Unknown acknowledgement"
        if "no matching data" in reason.lower() or "no data" in reason.lower():
            return []
        raise ValueError(f"ENTSO-E rejected a query: {reason}")
    rows = []
    for series in root.findall(".//{*}TimeSeries"):
        if (series.findtext("{*}out_Domain.mRID") != origin
                or series.findtext("{*}in_Domain.mRID") != destination
                or series.findtext("{*}quantity_Measure_Unit.name") != "MAW"):
            raise ValueError("Unexpected ENTSO-E bidding zones, direction or power unit.")
        curve = series.findtext("{*}curveType")
        if curve not in {"A01", "A03"}:
            raise ValueError("Unsupported ENTSO-E curve type.")
        for period in series.findall("{*}Period"):
            begin = pd.to_datetime(period.findtext("{*}timeInterval/{*}start"), utc=True)
            finish = pd.to_datetime(period.findtext("{*}timeInterval/{*}end"), utc=True)
            step = pd.Timedelta(period.findtext("{*}resolution"))
            if step not in (pd.Timedelta(minutes=15), pd.Timedelta(minutes=30), pd.Timedelta(hours=1)):
                raise ValueError("Unexpected ENTSO-E resolution.")
            points = period.findall("{*}Point")
            positions = [int(point.findtext("{*}position")) for point in points]
            if positions != sorted(set(positions)):
                raise ValueError("ENTSO-E positions must be ordered and unique.")
            # Interpreting A01 as fixed steps and A03 as values lasting until the next change.
            for i, point in enumerate(points):
                start = begin + (positions[i] - 1) * step
                end = (begin + (positions[i + 1] - 1) * step
                       if curve == "A03" and i + 1 < len(points)
                       else finish if curve == "A03" else start + step)
                power = float(point.findtext("{*}quantity"))
                if not begin <= start < end <= finish or not np.isfinite(power):
                    raise ValueError("Invalid ENTSO-E interval or power.")
                start, end = max(start, request_start), min(end, request_end)
                if start < end:
                    rows.append((start, end, power))
    return rows


def weekly_flow_energy(blocks):
    """Split non-overlapping UTC power intervals at local week boundaries."""
    blocks = blocks.sort_values("start")
    if blocks["start"].lt(blocks["end"].shift()).any():
        raise ValueError("Overlapping ENTSO-E blocks; do not sum duplicate publications.")
    pieces = []
    for start, end, power in blocks[["start", "end", "power"]].itertuples(index=False, name=None):
        # Splitting blocks crossing Monday midnight, accounting for daylight saving time.
        while start < end:
            label = start.tz_convert(LOCAL_ZONE).tz_localize(None).to_period("W-SUN").end_time.normalize()
            boundary = (label + pd.Timedelta(days=1)).tz_localize(LOCAL_ZONE).tz_convert("UTC")
            stop = min(end, boundary)
            hours = (stop - start) / pd.Timedelta(hours=1)
            pieces.append((label, power * hours, hours))
            start = stop
    if not pieces:
        return pd.Series(np.nan, index=weeks)
    totals = pd.DataFrame(pieces, columns=["date", "energy", "hours"]).groupby("date").sum().reindex(weeks)
    # Leaving incomplete weeks missing, including weeks around historical connection gaps.
    complete = np.isclose(totals["hours"], expected_week_hours)
    return totals["energy"].where(complete)


# Reading only NO3 neighbour files; empty responses remain explicitly documented.
flow_rows, missing_flow_queries = [], []
for filename, neighbour, direction, origin, destination, start, end in entsoe_files:
    rows = parse_flow_response((DATA_FOLDER / filename).read_text(encoding="utf-8"),
                               origin, destination, start, end)
    if not rows:
        missing_flow_queries.append((neighbour, direction, start))
    flow_rows.extend((neighbour, direction, *row) for row in rows)
flows = pd.DataFrame(flow_rows, columns=["neighbour", "direction", "start", "end", "power"])
if flows.empty:
    raise ValueError("No ENTSO-E physical-flow observations were returned.")
entsoe_weekly = pd.DataFrame(index=weeks)
for neighbour in neighbour_codes:
    for direction in ("import", "export"):
        selected = flows.loc[flows["neighbour"].eq(neighbour) & flows["direction"].eq(direction)]
        column = f"no3_{neighbour.lower()}_{direction}_mwh"
        entsoe_weekly[column] = weekly_flow_energy(selected)
print(f"ENTSO-E: {len(missing_flow_queries)} NO3 neighbour/direction/month queries without data.")
print("Missing weeks per feature:")
print(entsoe_weekly.isna().sum())
entsoe_weekly.head()


ENTSO-E: 2 NO3 neighbour/direction/month queries without data.
Missing weeks per feature:
no3_se2_import_mwh     0
no3_se2_export_mwh     0
no3_no5_import_mwh    21
no3_no5_export_mwh    21
no3_no1_import_mwh    16
no3_no1_export_mwh    16
no3_no4_import_mwh    16
no3_no4_export_mwh    16
dtype: int64


,no3_se2_import_mwh,no3_se2_export_mwh,no3_no5_import_mwh,no3_no5_export_mwh,no3_no1_import_mwh,no3_no1_export_mwh,no3_no4_import_mwh,no3_no4_export_mwh
date,,,,,,,,
2016-01-10,21789.89,5820.7,NaN,NaN,23403.97,35.74,121260.39,0.0
2016-01-17,8443.55,13821.4,NaN,NaN,14591.25,2612.79,138625.10,0.0
2016-01-24,140.23,22375.0,NaN,NaN,4199.07,6546.81,126269.12,0.0
2016-01-31,73941.14,2850.3,NaN,NaN,10628.95,7470.27,109352.74,0.0
2016-02-07,69057.72,1468.4,NaN,NaN,17404.87,4095.16,113919.02,0.0


### 6.4 Open-Meteo: round one, 2016-2020

We request hourly ERA5 temperature and precipitation for the same 100 NO3 points from the [Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api). Each year is saved separately and checked before we continue.

We split the download into two five-year rounds and leave at least 60 seconds between new annual requests. Long requests count as multiple calls under the [Open-Meteo quotas](https://open-meteo.com/en/pricing), so the number of HTTP requests alone does not describe quota use.


In [ ]:
# Reading only the fixed coordinates, independently of earlier weather datasets and app functions.
locations = np.asarray(json.loads((PROJECT_ROOT / "Compulsory Assignments/assets/weather_locations.json")
                                 .read_text(encoding="utf-8"))["regions"]["NO3"], dtype=float)
if locations.shape != (100, 2) or not np.isfinite(locations).all() or (np.abs(locations) > [180, 90]).any():
    raise ValueError("Expected 100 valid longitude/latitude pairs for NO3.")


# Recording the exact requested sample so later rounds cannot mix different coordinates.
location_file = DATA_FOLDER / "weather/NO3_locations.json"
if location_file.is_file():
    if json.loads(location_file.read_text(encoding="utf-8")) != locations.tolist():
        raise ValueError("The fixed NO3 sample differs from the sample saved for CA4.")
else:
    location_file.parent.mkdir(parents=True, exist_ok=True)
    location_file.write_text(json.dumps(locations.tolist()), encoding="utf-8")


def weather_hourly_mean(payload, year):
    """Validate 100 full-year responses and return equal-weight regional hourly means."""
    hours = pd.date_range(f"{year}-01-01", f"{year + 1}-01-01", freq="h", inclusive="left", tz="UTC")
    if not isinstance(payload, list) or len(payload) != 100:
        raise ValueError("Expected exactly 100 weather responses.")
    values = []
    for point in payload:
        frame = pd.DataFrame(point["hourly"])[["time", "temperature_2m", "precipitation"]]
        frame["time"] = pd.to_datetime(frame["time"], utc=True)
        if (not pd.DatetimeIndex(frame["time"]).equals(hours)
                or point.get("utc_offset_seconds") != 0
                or point["hourly_units"].get("temperature_2m") != "\u00b0C"
                or point["hourly_units"].get("precipitation") != "mm"):
            raise ValueError(f"Unexpected weather hours, timezone or units in {year}.")
        numeric = frame[["temperature_2m", "precipitation"]].to_numpy(dtype=float)
        if not np.isfinite(numeric).all() or (numeric[:, 1] < 0).any():
            raise ValueError(f"Missing/invalid weather values in {year}.")
        values.append(numeric)
    # Retaining identical grid series as separate sampled points with equal weights.
    return pd.DataFrame(np.mean(values, axis=0), index=hours,
                        columns=["temperature_c", "precipitation_mm"]).rename_axis("time")


def download_weather_years(years):
    """Save five annual CA4 weather responses, with 60-second request-start spacing."""
    files, last_start = [], None
    for year in years:
        filename = f"weather/NO3_{year}.json"
        path = DATA_FOLDER / filename
        if path.is_file():
            weather_hourly_mean(json.loads(path.read_text(encoding="utf-8")), year)
        else:
            if last_start is not None:
                time.sleep(max(0, 60 - (time.monotonic() - last_start)))
            print(f"Open-Meteo: requesting NO3/{year}.", flush=True)
            last_start = time.monotonic()
            payload = json.loads(download_text("https://archive-api.open-meteo.com/v1/archive",
                {"latitude": ",".join(map(str, locations[:, 1])),
                 "longitude": ",".join(map(str, locations[:, 0])),
                 "start_date": f"{year}-01-01", "end_date": f"{year}-12-31",
                 "hourly": "temperature_2m,precipitation", "models": "era5", "timezone": "GMT",
                 "temperature_unit": "celsius", "precipitation_unit": "mm"}, filename))
            try:
                weather_hourly_mean(payload, year)
            except (ValueError, KeyError, TypeError):
                # Removing only this invalid CA4 response so a later run can download it again.
                path.unlink()
                raise
        files.append(path)
        print(f"Open-Meteo: NO3/{year} validated and saved.", flush=True)
    return files


# Starting the first five-year round; running this cell makes live requests for missing files.
weather_round_one = download_weather_years(range(2016, 2021))



Open-Meteo: requesting NO3/2016.
Open-Meteo: NO3/2016 validated and saved.
Open-Meteo: requesting NO3/2017.
Open-Meteo: NO3/2017 validated and saved.
Open-Meteo: requesting NO3/2018.
Open-Meteo: NO3/2018 validated and saved.
Open-Meteo: requesting NO3/2019.
Open-Meteo: NO3/2019 validated and saved.
Open-Meteo: requesting NO3/2020.
Open-Meteo: NO3/2020 validated and saved.
Round one finished. Round two starts with an automatic 65-minute wait.


### 6.4 Open-Meteo: round two, 2021-2025

We download the remaining five years with the same points and settings. We had to wait for one hour after running the previous cell, before we did this.


In [20]:

weather_round_two = download_weather_years(range(2021, 2026))
print("Round two finished. Both rounds can now be processed together.")


Open-Meteo: requesting NO3/2021.
Open-Meteo: NO3/2021 validated and saved.
Open-Meteo: requesting NO3/2022.
Open-Meteo: NO3/2022 validated and saved.
Open-Meteo: requesting NO3/2023.
Open-Meteo: NO3/2023 validated and saved.
Open-Meteo: requesting NO3/2024.
Open-Meteo: NO3/2024 validated and saved.
Open-Meteo: requesting NO3/2025.
Open-Meteo: NO3/2025 validated and saved.
Round two finished. Both rounds can now be processed together.


### 6.4 Open-Meteo: concatenate and process both rounds

We combine all ten years and average the 100 points for each hour. We then calculate weekly mean temperature and weekly precipitation totals, with precipitation assigned to the hour it covers.

The result contains 87,672 UTC hours and 521 complete weeks. Precipitation is the average weekly amount across the points, not a sum across all 100 locations. We account for the shorter and longer weeks around daylight saving changes.


In [21]:
# Concatenating all ten validated annual regional series, regardless of notebook restarts.
weather_files = [DATA_FOLDER / f"weather/NO3_{year}.json" for year in range(2016, 2026)]
if not all(path.is_file() for path in weather_files):
    raise ValueError("Complete both download rounds before processing all ten years.")
weather_hourly = pd.concat([
    weather_hourly_mean(json.loads(path.read_text(encoding="utf-8")), year)
    for year, path in zip(range(2016, 2026), weather_files)
]).sort_index()
expected_hours = pd.date_range(START_DATE, END_DATE, freq="h", inclusive="left", tz="UTC")
if not weather_hourly.index.equals(expected_hours):
    raise ValueError("Weather years have gaps, overlaps or unexpected time coverage.")

# Matching local weeks: temperature is instantaneous; precipitation ends at its timestamp.
local_times = weather_hourly.index.tz_convert(LOCAL_ZONE)
temperature_labels = local_times.tz_localize(None).to_period("W-SUN").end_time.normalize()
rain_labels = (local_times - pd.Timedelta(1, unit="ns")).tz_localize(None).to_period("W-SUN").end_time.normalize()
temperature = weather_hourly["temperature_c"].groupby(temperature_labels).agg(["mean", "count"])
precipitation = weather_hourly["precipitation_mm"].groupby(rain_labels).agg(["sum", "count"])
if (not temperature["count"].reindex(weeks).eq(expected_week_hours).all()
        or not precipitation["count"].reindex(weeks).eq(expected_week_hours).all()):
    raise ValueError("Incomplete weather weeks; daylight-saving weeks may contain 167 or 169 hours.")
weather_weekly = pd.DataFrame({"temperature_c": temperature["mean"],
                               "precipitation_mm": precipitation["sum"]}).reindex(weeks)
print(f"Weather: {len(weather_hourly)} UTC hours; {len(weather_weekly)} complete local weeks.")
weather_weekly.head()


Weather: 87672 UTC hours; 521 complete local weeks.


,temperature_c,precipitation_mm
date,,
2016-01-10,-9.976821,1.154
2016-01-17,-6.075530,13.761
2016-01-24,-2.447381,25.797
2016-01-31,1.850619,59.866
2016-02-07,0.437679,54.601


## 7. Data coverage and quality

We first check that all four tables have the same 521 ordered weekly dates, numeric measurements and reasonable ranges. We allow negative temperatures, but not negative precipitation or directional flow energy.

We then look more closely at incomplete flow weeks. The second table shows how many hours are missing, so we can distinguish a short gap from an entirely missing week.



In [22]:
# Checking weekly alignment before joining, preventing duplicate dates and shifted weeks.
weekly_tables = {"NVE": reservoir_weekly, "NB": currency_weekly,
                 "ENTSO-E": entsoe_weekly, "Weather": weather_weekly}
coverage_rows = []
for source, table in weekly_tables.items():
    assert len(table) == 521 and table.index.is_unique and table.index.equals(weeks), (
        f"{source}: expected the same 521 unique, ordered weekly dates.")
    for column in table.columns:
        values = table[column]
        if column != "area_type":
            assert pd.api.types.is_numeric_dtype(values), f"{column}: expected numeric values."
            assert np.isfinite(values.dropna()).all(), f"{column}: infinite values found."
        coverage_rows.append({"source": source, "column": column, "dtype": str(values.dtype),
                              "missing_weeks": values.isna().sum(),
                              "first_valid": values.first_valid_index(),
                              "last_valid": values.last_valid_index(),
                              "min": values.min(), "max": values.max()})

# Checking obvious region/unit mistakes without treating sub-zero temperatures as errors.
assert reservoir_weekly["area_type"].eq("EL").all() and reservoir_weekly["area_number"].eq(3).all()
assert reservoir_weekly["filling_percent"].between(0, 100).all()
assert currency_weekly.gt(0).all().all(), "Exchange rates must be positive."
assert weather_weekly["precipitation_mm"].ge(0).all(), "Precipitation cannot be negative."
assert (entsoe_weekly.ge(0) | entsoe_weekly.isna()).all().all(), "Inspect negative directional flows."
assert not any(table.isna().any().any() for source, table in weekly_tables.items()
               if source != "ENTSO-E"), "Unexpected gaps outside the flow data."
display(pd.DataFrame(coverage_rows))


,source,column,dtype,missing_weeks,first_valid,last_valid,min,max
0,NVE,area_type,str,0,2016-01-10,2025-12-28,EL,EL
1,NVE,area_number,int64,0,2016-01-10,2025-12-28,3,3
2,NVE,iso_year,int64,0,2016-01-10,2025-12-28,2016,2025
3,NVE,iso_week,int64,0,2016-01-10,2025-12-28,1,53
4,NVE,filling_percent,float64,0,2016-01-10,2025-12-28,10.903239,96.095973
5,NVE,stored_energy_twh,float64,0,2016-01-10,2025-12-28,0.972671,8.572657
6,NVE,capacity_twh,float64,0,2016-01-10,2025-12-28,8.920932,8.920932
7,NVE,previous_week_filling_percent,float64,0,2016-01-10,2025-12-28,10.903239,96.094954
8,NVE,weekly_change_pp,float64,0,2016-01-10,2025-12-28,-5.148226,17.145711
9,NB,eur_nok,float64,0,2016-01-10,2025-12-28,8.82588,12.021867


We see complete reservoir, currency and weather columns, while some flow columns have missing weeks. The next cell checks whether those missing totals represent a few hours or much longer gaps.


In [25]:
# Measuring interval coverage only for missing weekly values; retaining all original data.
flow_gap_rows = []
for date in entsoe_weekly.index[entsoe_weekly.isna().any(axis=1)]:
    position = weeks.get_loc(date)
    start, end = week_starts[position], week_ends[position]
    for neighbour in neighbour_codes:
        for direction in ("import", "export"):
            column = f"no3_{neighbour.lower()}_{direction}_mwh"
            if pd.isna(entsoe_weekly.loc[date, column]):
                # Clipping published intervals to this week, including partial boundary hours.
                blocks = flows.loc[(flows["neighbour"] == neighbour) & (flows["direction"] == direction)
                                   & (flows["start"] < end) & (flows["end"] > start)]
                covered = ((blocks["end"].clip(upper=end) - blocks["start"].clip(lower=start))
                           .dt.total_seconds() / 3600).sum()
                expected = expected_week_hours.loc[date]
                flow_gap_rows.append({"date": date, "expected_hours": expected,
                                      "covered_hours": covered, "missing_hours": expected - covered,
                                      "series": column})

# Combining identical gaps into fewer rows while keeping the affected series visible.
flow_gap_hours = pd.DataFrame(flow_gap_rows, columns=[
    "date", "expected_hours", "covered_hours", "missing_hours", "series"])
display(flow_gap_hours.groupby(["date", "expected_hours", "covered_hours", "missing_hours"],
                              as_index=False).agg({"series": ", ".join}))


,date,expected_hours,covered_hours,missing_hours,series
0,2016-01-10,168.0,0.0,168.0,"no3_no5_import_mwh, no3_no5_export_mwh"
1,2016-01-17,168.0,0.0,168.0,"no3_no5_import_mwh, no3_no5_export_mwh"
2,2016-01-24,168.0,0.0,168.0,"no3_no5_import_mwh, no3_no5_export_mwh"
3,2016-01-31,168.0,0.0,168.0,"no3_no5_import_mwh, no3_no5_export_mwh"
4,2016-02-07,168.0,0.0,168.0,"no3_no5_import_mwh, no3_no5_export_mwh"
5,2016-02-14,168.0,0.0,168.0,"no3_no5_import_mwh, no3_no5_export_mwh"
6,2016-02-14,168.0,164.0,4.0,"no3_no1_import_mwh, no3_no1_export_mwh, no3_no..."
7,2016-07-24,168.0,129.0,39.0,"no3_no5_import_mwh, no3_no5_export_mwh, no3_no..."
8,2016-10-30,169.0,167.0,2.0,"no3_no5_import_mwh, no3_no5_export_mwh, no3_no..."
9,2016-11-06,168.0,167.0,1.0,"no3_no5_import_mwh, no3_no5_export_mwh, no3_no..."


We see no missing weeks in reservoir, currency or weather data. The flow gaps range from single hours to entire weeks, so we keep all 521 dates and leave incomplete flow totals as `NaN` in the combined data.


## 8. Combining the weekly data

We join the other sources to the reservoir table using the weekly date. The checks confirm that we still have 521 weeks and 21 columns, with the original values and missing flow values preserved. This gives us one analysis table; we select the model inputs separately.



In [26]:
# Joining on the weekly date index, keeping the reservoir weeks and rejecting duplicate matches.
no3_weekly = (
    reservoir_weekly.join(currency_weekly, how="left", validate="one_to_one")
    .join(entsoe_weekly, how="left", validate="one_to_one")
    .join(weather_weekly, how="left", validate="one_to_one")
)

# Verifying that the join preserves dates, all source values and existing missing flow values.
assert no3_weekly.shape == (521, 21) and no3_weekly.index.equals(weeks)
for table in (reservoir_weekly, currency_weekly, entsoe_weekly, weather_weekly):
    pd.testing.assert_frame_equal(no3_weekly[table.columns], table)
print(f"Combined NO3 data: {len(no3_weekly)} weeks and {len(no3_weekly.columns)} columns.")
no3_weekly.head()


Combined NO3 data: 521 weeks and 21 columns.


,area_type,area_number,iso_year,iso_week,filling_percent,stored_energy_twh,capacity_twh,previous_week_filling_percent,weekly_change_pp,eur_nok,...,no3_se2_import_mwh,no3_se2_export_mwh,no3_no5_import_mwh,no3_no5_export_mwh,no3_no1_import_mwh,no3_no1_export_mwh,no3_no4_import_mwh,no3_no4_export_mwh,temperature_c,precipitation_mm
date,,,,,,,,,,,,,,,,,,,,,
2016-01-10,EL,3,2016,1,68.995315,6.155025,8.920932,72.939220,-3.943902,9.64730,...,21789.89,5820.7,NaN,NaN,23403.97,35.74,121260.39,0.0,-9.976821,1.154
2016-01-17,EL,3,2016,2,64.587426,5.761800,8.920932,68.995315,-4.407889,9.61804,...,8443.55,13821.4,NaN,NaN,14591.25,2612.79,138625.10,0.0,-6.075530,13.761
2016-01-24,EL,3,2016,3,59.888100,5.342576,8.920932,64.587426,-4.699326,9.62910,...,140.23,22375.0,NaN,NaN,4199.07,6546.81,126269.12,0.0,-2.447381,25.797
2016-01-31,EL,3,2016,4,57.638526,5.141893,8.920932,59.888100,-2.249575,9.46522,...,73941.14,2850.3,NaN,NaN,10628.95,7470.27,109352.74,0.0,1.850619,59.866
2016-02-07,EL,3,2016,5,54.853940,4.893482,8.920932,57.638526,-2.784586,9.51122,...,69057.72,1468.4,NaN,NaN,17404.87,4095.16,113919.02,0.0,0.437679,54.601


## 9. First River forecasting experiment

### Model and scope

We start with SNARIMAX in River and a linear regression model. `p` chooses recent filling lags, `sp=1` adds a seasonal lag, and `sd=1` makes the model learn seasonal differences. The inputs are standardized before regression.

The function trains on the selected history up to each forecast origin, meaning the last observed week. River then uses its own predictions to make the next steps, without receiving future observations. `learning_rate` controls how much the weights change during training. We limit the returned filling values to 0-100% after forecasting; this does not change the internal calculations.

We begin with reservoir history only, then add source choices and compare history lengths and time resolutions. These are retrospective tests using observation dates; publication delays are not simulated.



In [60]:
# Defining a small NO3-only forecast with explicit training and model settings.
from river import linear_model, optim, preprocessing, time_series


def normal_weather_scenario(history_weather, dates):
    """Use training-history ISO-week means as a NO3 weather scenario, not a weather forecast."""
    # Averaging only past observations; keeping precipitation as a weekly total in mm.
    means = history_weather.groupby(history_weather.index.isocalendar().week.to_numpy()).mean()
    scenario = means.reindex(dates.isocalendar().week.to_numpy()).set_axis(dates)
    if scenario.isna().any().any():
        raise ValueError("No training-weather mean for a forecast ISO week; inspect week 53 if relevant.")
    return scenario


def four_week_inputs(table):
    """Aggregate four weekly inputs, keeping rates/temperature as means and energy/rain as sums."""
    # Using blocks ending on the existing Sunday labels, rather than inventing calendar months.
    if not len(table.columns):
        return table.iloc[3::4]
    rules = {c: "sum" if c == "precipitation_mm" or c.endswith("_mwh") else "mean" for c in table}
    return table.groupby(np.arange(len(table)) // 4).agg(rules).set_axis(table.index[3::4])


def forecast_no3(data, origin, p=4, sp=0, history_weeks=260, horizon_weeks=13,
                 learning_rate=0.01, sd=0, use_weather=False, use_currency=False, use_flows=False,
                 step_weeks=1):
    """Forecast NO3 with selected sources at weekly or four-week intervals."""
    # Requiring whole blocks and enough history for seasonal differencing and lag inputs.
    if step_weeks not in (1, 4) or history_weeks % step_weeks or horizon_weeks % step_weeks:
        raise ValueError("Use step_weeks=1 or 4 and history/horizon divisible by that step.")
    season = 52 // step_weeks
    history = data.loc[:origin, "filling_percent"].tail(history_weeks)
    if sd not in (0, 1) or history_weeks // step_weeks <= season * sd + max(p, season * sp) or horizon_weeks < 1:
        raise ValueError("Use sd=0 or 1, sufficient history for differencing/lags, and a positive horizon.")
    if len(history) != history_weeks or history.isna().any():
        raise ValueError("The training history must contain all requested weekly values.")

    # Selecting sources and filling missing flow inputs from earlier observations in a copy.
    weather_columns = ["temperature_c", "precipitation_mm"] if use_weather else []
    currency_columns = ["eur_nok", "usd_nok"] if use_currency else []
    flow_columns = [c for c in data if c.startswith("no3_") and c.endswith("_mwh")] if use_flows else []
    if use_flows and len(flow_columns) != 8:
        raise ValueError("Expected the eight NO3 neighbour-flow columns.")
    columns = weather_columns + currency_columns + flow_columns
    inputs = data.loc[:origin, columns].copy()
    inputs[flow_columns] = inputs[flow_columns].ffill()
    inputs = inputs.loc[history.index]
    if inputs.isna().any().any():
        raise ValueError("An input has no past value; choose a later origin or shorter history.")

    # Building weekly scenarios first, preserving the original calendar-week weather assumptions.
    dates = pd.date_range(history.index[-1] + pd.Timedelta(weeks=1),
                          periods=horizon_weeks, freq="W-SUN", name="date")
    future = pd.DataFrame({c: inputs[c].iloc[-1] for c in columns}, index=dates)
    if use_weather:
        future[weather_columns] = normal_weather_scenario(inputs[weather_columns], dates)
    if step_weeks == 4:
        # Keeping filling as an end-of-period snapshot; aggregating the other input quantities.
        history = history.groupby(np.arange(len(history)) // 4).last().set_axis(history.index[3::4])
        inputs, future = four_week_inputs(inputs), four_week_inputs(future)
        dates = dates[3::4]

    # Learning on levels or seasonal differences, then forecasting recursively without future truth.
    initial_mean = history.diff(season).dropna().mean() if sd else history.mean()
    regressor = preprocessing.StandardScaler() | linear_model.LinearRegression(
        optimizer=optim.SGD(learning_rate), intercept_init=float(initial_mean))
    model = time_series.SNARIMAX(p=p, d=0, q=0, m=season, sp=sp, sd=sd, sq=0, regressor=regressor)
    for date, filling in history.items():
        model.learn_one(float(filling), x=inputs.loc[date].to_dict() if columns else None)
    forecast = pd.Series(model.forecast(horizon=len(dates),
                         xs=future.to_dict("records") if columns else None), index=dates, name="prediction")
    # Applying physical bounds to returned predictions; keeping River's internal recursion unchanged.
    return forecast.clip(lower=0, upper=100), model


### Comparing six settings

We test `p=4, 8, 13`, with and without the 52-week seasonal lag (`sp=0, 1`). Each model uses 260 weeks of history and predicts 13 weeks ahead at the same eight origins in 2023-2024. We leave 2025 out of this selection.

We measure MAE in percentage points over 104 predictions per setting. We compare with repeating the last observed filling and with using the filling from 52 weeks earlier. The timing covers preparation, training and forecasting, excluding imports, downloads and plots.


In [61]:
# Comparing all settings at the same chronological origins, leaving 2025 unused.
from time import perf_counter

history_weeks, horizon_weeks = 260, 13
origins = pd.date_range("2023-01-01", "2024-09-29", freq="13W-SUN")
experiment_rows, forecast_paths = [], []
for origin in origins:
    dates = pd.date_range(origin + pd.Timedelta(weeks=1), periods=horizon_weeks, freq="W-SUN")
    actual = no3_weekly.loc[dates, "filling_percent"]
    persistence = no3_weekly.loc[origin, "filling_percent"]
    seasonal_reference = no3_weekly.loc[dates - pd.Timedelta(weeks=52), "filling_percent"].to_numpy()
    for p in (4, 8, 13):
        for sp in (0, 1):
            started = perf_counter()
            forecast, model = forecast_no3(no3_weekly, origin, p=p, sp=sp,
                                          history_weeks=history_weeks, horizon_weeks=horizon_weeks)
            elapsed = perf_counter() - started
            # Measuring forecasts against truth only after the whole horizon has been predicted.
            experiment_rows.append({"origin": origin, "p": p, "sp": sp,
                "mae_pp": float((actual - forecast).abs().mean()),
                "persistence_mae_pp": float((actual - persistence).abs().mean()),
                "seasonal_reference_mae_pp": float(np.abs(actual.to_numpy() - seasonal_reference).mean()),
                "seconds": elapsed, "outside_0_100": int((~forecast.between(0, 100)).sum())})
            forecast_paths.append(pd.DataFrame({"origin": origin, "setting": f"p={p}, sp={sp}",
                                                "prediction": forecast}))

# Summarizing errors and runtime without selecting settings on the reserved 2025 data.
experiment_results = pd.DataFrame(experiment_rows)
comparison = experiment_results.groupby(["p", "sp"]).agg(
    mae_pp=("mae_pp", "mean"), persistence_mae_pp=("persistence_mae_pp", "mean"),
    seasonal_reference_mae_pp=("seasonal_reference_mae_pp", "mean"),
    mean_seconds=("seconds", "mean"), outside_0_100=("outside_0_100", "sum"),
).sort_values("mae_pp")
forecast_paths = pd.concat(forecast_paths)
display(comparison.round(4))


mae_pp  persistence_mae_pp  seasonal_reference_mae_pp  mean_seconds  \
p  sp                                                                         
4  1   10.9131             14.1916                     7.8931        0.0088   
8  1   11.2485             14.1916                     7.8931        0.0103   
13 1   11.7887             14.1916                     7.8931        0.0127   
   0   13.7294             14.1916                     7.8931        0.0157   
8  0   14.7320             14.1916                     7.8931        0.0094   
4  0   15.1854             14.1916                     7.8931        0.0083   

       outside_0_100  
p  sp                 
4  1               0  
8  1               0  
13 1               0  
   0               0  
8  0               0  
4  0               0

### Inspecting the selected River candidate

Here we inspect the best River setting from the first comparison. The table shows the lag values, their standardized values and their weighted contributions. Adding the intercept gives the first raw prediction, before the 0-100% limit. These weights describe the model, not causal effects.

The plot shows one 13-week period, while the score above averages all eight periods. We therefore need both the plot and the average error when judging the result.


In [62]:
# Exposing the lag inputs and learned contributions for the first forecast step.
best_p, best_sp = map(int, comparison.index[0])
origin = origins[-1]
forecast, model = forecast_no3(no3_weekly, origin, p=best_p, sp=best_sp,
                              history_weeks=history_weeks, horizon_weeks=horizon_weeks)
history = no3_weekly.loc[:origin, "filling_percent"].tail(history_weeks)
lag_values = {f"y-{lag}": float(history.iloc[-lag]) for lag in range(1, best_p + 1)}
if best_sp:
    lag_values["sy-52"] = float(history.iloc[-52])
scaled_values = model.regressor["StandardScaler"].transform_one(lag_values)
weights = model.regressor["LinearRegression"].weights
lag_table = pd.DataFrame({"filling_percent": pd.Series(lag_values),
                          "standardized_input": pd.Series(scaled_values),
                          "weight": pd.Series(weights).reindex(lag_values)})
lag_table["contribution_pp"] = lag_table["standardized_input"] * lag_table["weight"]
intercept = model.regressor["LinearRegression"].intercept
raw_first = intercept + lag_table["contribution_pp"].sum()
print(f"First prediction before/after the 0-100% limit: {raw_first:.3f}% / {forecast.iloc[0]:.3f}%")
display(lag_table.round(3))

# Plotting the same held-out dates for all models; legend clicks can hide individual lines.
import plotly.express as px

plot_data = forecast_paths.loc[forecast_paths["origin"].eq(origin)].pivot(
    columns="setting", values="prediction")
plot_data["Observed"] = no3_weekly.loc[plot_data.index, "filling_percent"]
plot_data["Persistence"] = no3_weekly.loc[origin, "filling_percent"]
plot_data["52-week reference"] = no3_weekly.loc[
    plot_data.index - pd.Timedelta(weeks=52), "filling_percent"].to_numpy()
figure = px.line(plot_data, title=f"NO3: 13-week forecasts after {origin.date()}",
                 labels={"value": "Filling (%)", "date": "Week ending", "variable": "Forecast"})
figure.update_traces(selector={"name": "Observed"}, line={"color": "black", "width": 4})
figure.update_layout(template="plotly_white", hovermode="x unified", legend_title_text="Forecast")
figure


First prediction before/after the 0-100% limit: 84.572% / 84.572%


,filling_percent,standardized_input,weight,contribution_pp
y-1,85.250,1.105,9.280,10.250
y-2,85.228,1.108,5.674,6.289
y-3,86.205,1.156,2.383,2.754
y-4,82.801,1.013,-0.498,-0.504
sy-52,83.506,0.989,3.222,3.187


### Findings from the first comparison

We see the lowest River MAE with `p=4, sp=1`: **10.91 pp**. This is better than repeating the last filling (**14.19 pp**), but worse than the 52-week reference (**7.89 pp**). Adding more recent lags does not improve this seasonal model, so we next try learning the difference from last year.


### One follow-up: forecasting deviations from last year

We now test `p=4, sp=0, sd=1` with the same history, origins and horizon. Instead of predicting the full level directly, the model learns `filling[t] - filling[t-52]`, and River adds the seasonal level back afterwards. The first 52 observations establish the seasonal history, leaving 208 possible training updates.

Course: [SARIMAX](../../Curriculum/D2Dbook/4_Machine_Learning/2_Forecasting/3_SARIMAX.ipynb), "SARIMA - seasonal autoregressive integrated moving average models"; [Streaming models](../../Curriculum/D2Dbook/4_Machine_Learning/3_Streaming/2_Streaming_models.ipynb), "Streaming forecasts".


In [63]:
# Testing one seasonal-difference model on the existing chronological validation periods.
seasonal_test_rows = []
for origin in origins:
    started = perf_counter()
    forecast, model = forecast_no3(no3_weekly, origin, p=4, sp=0, sd=1,
                                  history_weeks=history_weeks, horizon_weeks=horizon_weeks)
    elapsed = perf_counter() - started
    actual = no3_weekly.loc[forecast.index, "filling_percent"]
    seasonal_test_rows.append({"mae_pp": float((actual - forecast).abs().mean()),
                              "seconds": elapsed,
                              "outside_0_100": int((~forecast.between(0, 100)).sum())})

# Comparing mean errors; the reference predicts the earlier value without a learned correction.
seasonal_test_results = pd.DataFrame(seasonal_test_rows)
seasonal_comparison = pd.DataFrame({"mae_pp": [
    seasonal_test_results["mae_pp"].mean(), comparison.iloc[0]["mae_pp"],
    comparison.iloc[0]["seasonal_reference_mae_pp"]]},
    index=["River: p=4, sp=0, sd=1", "Previous best River: p=4, sp=1, sd=0", "52-week reference"])
display(seasonal_comparison.round(3))
print(f"Seasonal-difference model: {seasonal_test_results['seconds'].mean():.4f} seconds per origin; "
      f"{seasonal_test_results['outside_0_100'].sum()} forecasts outside 0-100%.")


,mae_pp
"River: p=4, sp=0, sd=1",8.251
"Previous best River: p=4, sp=1, sd=0",10.913
52-week reference,7.893


Seasonal-difference model: 0.0077 seconds per origin; 0 forecasts outside 0-100%.


### Seasonal-difference findings

We see that seasonal differencing reduces River's MAE from **10.91 to 8.25 pp**. The 52-week reference is still slightly better at **7.89 pp**. We keep the seasonal-difference model and next test whether weather adds useful information.


## 10. Adding weather under a normal-weather scenario

We add weekly temperature and precipitation to `p=4, sp=0, sd=1`, keeping the same eight origins, 260-week history and 13-week horizon. During training, each filling observation is paired with weather from the same week.

We do not know the weather 13 weeks ahead. We therefore use the mean for each future ISO week from the selected training history. This is a normal-weather scenario, not a weather forecast or an official climate normal. A missing week mean, including week 53, raises an error. We never give the model actual future weather.


In [64]:
# Comparing weather source selection while keeping all other model/evaluation choices fixed.
weather_test_rows, weather_paths = [], []
for origin in origins:
    for use_weather in (False, True):
        started = perf_counter()
        forecast, model = forecast_no3(no3_weekly, origin, p=4, sp=0, sd=1,
            history_weeks=history_weeks, horizon_weeks=horizon_weeks, use_weather=use_weather)
        elapsed = perf_counter() - started
        actual = no3_weekly.loc[forecast.index, "filling_percent"]
        setting = "With weather scenario" if use_weather else "Without weather"
        weather_test_rows.append({"origin": origin, "setting": setting,
            "mae_pp": float((actual - forecast).abs().mean()), "seconds": elapsed,
            "outside_0_100": int((~forecast.between(0, 100)).sum())})
        weather_paths.append(pd.DataFrame({"origin": origin, "setting": setting, "prediction": forecast}))

# Reporting errors and runtime, retaining the strong seasonal reference for context.
weather_test_results = pd.DataFrame(weather_test_rows)
weather_comparison = weather_test_results.groupby("setting").agg(
    mae_pp=("mae_pp", "mean"), mean_seconds=("seconds", "mean"),
    outside_0_100=("outside_0_100", "sum"))
display(weather_comparison.round(4))
print(f"52-week reference MAE: {comparison.iloc[0]['seasonal_reference_mae_pp']:.3f} pp")
weather_paths = pd.concat(weather_paths)


,mae_pp,mean_seconds,outside_0_100
setting,,,
With weather scenario,7.8595,0.0323,0
Without weather,8.2509,0.0068,0


52-week reference MAE: 7.893 pp


### Inspecting the assumed future weather

This table shows the 13 weekly weather values we assume at the last forecast origin. The plot compares the models on those same dates. It illustrates one period, so it can look different from the average result across all eight periods.


In [65]:
# Showing the weather inputs supplied for the last 13-week forecast, using training data only.
origin = origins[-1]
last_paths = weather_paths.loc[weather_paths["origin"].eq(origin)].pivot(
    columns="setting", values="prediction")
training_weather = no3_weekly.loc[:origin, ["temperature_c", "precipitation_mm"]].tail(history_weeks)
weather_scenario = normal_weather_scenario(training_weather, last_paths.index)
display(weather_scenario.round(2))

# Comparing observed filling with both River forecasts and the seasonal reference.
last_paths["Observed"] = no3_weekly.loc[last_paths.index, "filling_percent"]
last_paths["52-week reference"] = no3_weekly.loc[
    last_paths.index - pd.Timedelta(weeks=52), "filling_percent"].to_numpy()
weather_figure = px.line(last_paths, title=f"NO3: normal-weather scenario after {origin.date()}",
    labels={"value": "Filling (%)", "date": "Week ending", "setting": "Forecast"})
weather_figure.update_traces(selector={"name": "Observed"}, line={"color": "black", "width": 4})
weather_figure.update_layout(template="plotly_white", hovermode="x unified", legend_title_text="Forecast")
weather_figure


,temperature_c,precipitation_mm
date,,
2024-10-06,9.28,52.45
2024-10-13,6.24,54.99
2024-10-20,4.42,32.67
2024-10-27,5.01,34.62
2024-11-03,4.09,22.78
2024-11-10,2.57,33.94
2024-11-17,1.30,17.04
2024-11-24,0.76,47.19
2024-12-01,-2.45,18.40


### Weather experiment findings

We see a small average improvement, from **8.25 to 7.86 pp**, with weather. The 52-week reference scores **7.89 pp**, so the difference from that reference is very small. In the last plotted period, weather does not help: MAE is **17.88 pp**, compared with **17.80 pp** without weather. We keep weather as a candidate and test the other sources next.


## 11. Comparing data sources

We now compare reservoir history alone, weather, weather with currency, weather with flows, and all sources together. We keep the model settings and forecast periods unchanged, so only source selection changes.

For future currency and flow inputs, we repeat the last available values. For flows, this means fixed weekly MWh totals, not fixed MW. Missing historical flow inputs are filled from the most recent earlier value in a model copy; the combined source table keeps its `NaN` values. This simple assumption can be inaccurate for long gaps.

`filled_flow_values` counts filled week/series inputs across all eight training windows. Since those windows overlap, the same missing value can be counted more than once. We show errors for each period as well as the average.



In [66]:
# Comparing selected sources on identical forecast periods and model settings.
source_settings = {
    "Reservoir only": {}, "Weather": {"use_weather": True},
    "Weather + currency": {"use_weather": True, "use_currency": True},
    "Weather + flows": {"use_weather": True, "use_flows": True},
    "All sources": {"use_weather": True, "use_currency": True, "use_flows": True},
    "52-week reference": None,
}
source_rows = []
flow_columns = [c for c in no3_weekly if c.endswith("_mwh")]
for origin in origins:
    dates = pd.date_range(origin + pd.Timedelta(weeks=1), periods=horizon_weeks, freq="W-SUN")
    actual = no3_weekly.loc[dates, "filling_percent"].to_numpy()
    for setting, options in source_settings.items():
        started = perf_counter()
        if options is None:
            prediction = no3_weekly.loc[dates - pd.Timedelta(weeks=52), "filling_percent"].to_numpy()
        else:
            forecast, model = forecast_no3(no3_weekly, origin, p=4, sp=0, sd=1,
                history_weeks=history_weeks, horizon_weeks=horizon_weeks, **options)
            prediction = forecast.to_numpy()
        elapsed = perf_counter() - started
        filled = no3_weekly.loc[:origin, flow_columns].tail(history_weeks).isna().sum().sum()             if options and options.get("use_flows") else 0
        source_rows.append({"origin": origin, "setting": setting,
            "mae_pp": float(np.abs(actual - prediction).mean()), "seconds": elapsed,
            "filled_flow_values": filled, "outside_0_100": int(((prediction < 0) | (prediction > 100)).sum())})

# Reporting average and per-period errors, so the selected source effect remains visible.
source_results = pd.DataFrame(source_rows)
source_comparison = source_results.groupby("setting").agg(mae_pp=("mae_pp", "mean"),
    mean_seconds=("seconds", "mean"), filled_flow_values=("filled_flow_values", "sum"),
    outside_0_100=("outside_0_100", "sum")).sort_values("mae_pp")
display(source_comparison.round(3))
display(source_results.pivot(index="origin", columns="setting", values="mae_pp").round(2))


,mae_pp,mean_seconds,filled_flow_values,outside_0_100
setting,,,,
All sources,7.607,0.048,408,0
Weather,7.860,0.032,0,0
52-week reference,7.893,0.001,0,0
Reservoir only,8.251,0.007,0,0
Weather + currency,8.372,0.034,0,0
Weather + flows,10.226,0.044,408,0


setting,52-week reference,All sources,Reservoir only,Weather,Weather + currency,Weather + flows
origin,,,,,,
2023-01-01,9.21,6.27,10.81,9.82,4.87,11.36
2023-04-02,8.70,7.27,7.75,6.39,6.94,5.51
2023-07-02,10.04,10.12,9.39,6.42,19.11,5.23
2023-10-01,3.08,7.48,6.23,6.07,2.54,18.74
2023-12-31,1.51,4.44,3.84,5.07,6.03,8.15
2024-03-31,8.91,8.46,8.92,9.77,10.27,10.28
2024-06-30,3.54,2.61,1.26,1.45,1.27,7.64
2024-09-29,18.15,14.19,17.80,17.88,15.94,14.90


### Source comparison findings

We get the lowest average MAE with all sources: **7.61 pp**, compared with **7.86 pp** for weather and **7.89 pp** for the 52-week reference. Currency and flows do not improve the weather model when added separately, so more inputs are not automatically better. We carry all sources forward and compare history length and time resolution next.


## 12. History length and time resolution

We compare 2, 3 and 5 years of history with weekly and four-week inputs, keeping all sources. Four-week blocks end at the forecast origin: we keep the last filling observation, average weekly temperature and currency values, and sum precipitation and flow energy. These are four-week periods rather than calendar months.

We use `p=4, m=52` for weekly data and `p=1, m=13` for four-week data, with `sp=0, sd=1` and learning rate 0.01. Future scenarios and flow-gap handling are prepared weekly before aggregation.

Both models are scored 4, 8 and 12 weeks ahead at the same eight origins, giving 24 common target dates. These MAEs therefore use different scoring dates from the earlier 13-week tables. We also compare runtime to check whether the model is practical for the app.

Course: [Synchronisation](../../Curriculum/D2Dbook/3_Data_quality/3_Preprocessing/4_Synchronisation.ipynb), "Accumulation"; [ML approach](../../Curriculum/D2Dbook/4_Machine_Learning/2_Forecasting/2_ML_approach.ipynb), "Validation" and "Backtesting".


In [67]:
# Comparing six small setups at the same three forecast endpoints per origin.
resolution_rows = []
for history_years in (2, 3, 5):
    for step_weeks in (1, 4):
        for origin in origins:
            dates = pd.DatetimeIndex([origin + pd.Timedelta(weeks=h) for h in (4, 8, 12)])
            started = perf_counter()
            forecast, model = forecast_no3(no3_weekly, origin, p=4 if step_weeks == 1 else 1,
                sd=1, history_weeks=52 * history_years, horizon_weeks=12, step_weeks=step_weeks,
                use_weather=True, use_currency=True, use_flows=True)
            elapsed = perf_counter() - started
            actual = no3_weekly.loc[dates, "filling_percent"].to_numpy()
            predicted = forecast.loc[dates].to_numpy()
            reference = no3_weekly.loc[dates - pd.Timedelta(weeks=52), "filling_percent"].to_numpy()
            resolution_rows.append({"history_years": history_years, "step_weeks": step_weeks,
                "origin": origin, "mae_pp": float(np.abs(actual - predicted).mean()),
                "reference_mae_pp": float(np.abs(actual - reference).mean()), "seconds": elapsed})

# Ranking the tested candidates and exposing the settings for the later reserved-year test.
resolution_results = pd.DataFrame(resolution_rows)
resolution_comparison = resolution_results.groupby(["history_years", "step_weeks"]).agg(
    mae_pp=("mae_pp", "mean"), reference_mae_pp=("reference_mae_pp", "mean"),
    mean_seconds=("seconds", "mean")).sort_values("mae_pp")
display(resolution_comparison.round(4))
best_history, best_step = map(int, resolution_comparison.index[0])
candidate_settings = dict(p=4 if best_step == 1 else 1, sp=0, sd=1,
    history_weeks=52 * best_history, horizon_weeks=12, step_weeks=best_step,
    use_weather=True, use_currency=True, use_flows=True)
print("Candidate for the 2025 test:", candidate_settings)


,,mae_pp,reference_mae_pp,mean_seconds
history_years,step_weeks,,,
2,4,6.8773,7.7878,0.0211
5,4,7.0244,7.7878,0.0273
3,4,7.2722,7.7878,0.0268
2,1,8.0010,7.7878,0.0250
5,1,8.1463,7.7878,0.0465
3,1,8.8224,7.7878,0.0350


Candidate for the 2025 test: {'p': 1, 'sp': 0, 'sd': 1, 'history_weeks': 104, 'horizon_weeks': 12, 'step_weeks': 4, 'use_weather': True, 'use_currency': True, 'use_flows': True}


### Resolution/history findings

We see the lowest tested MAE with **2 years of history and four-week steps: 6.88 pp**, compared with **7.79 pp** for the 52-week reference on the same dates. Average runtime ranges from about **0.021 to 0.047 seconds per origin**, so speed is not a problem for any of these settings.

We choose this setup as the candidate for 2025. It has only 26 four-week observations and 13 possible updates after seasonal differencing, so the next evaluation is important for checking how it works on another year.


## 13. Evaluation on 2025

We now evaluate the selected setup without changing its model settings: all sources, 2 years of history, four-week steps and `p=1, sp=0, sd=1`. We use four origins from 2024-12-29 to 2025-09-28 and score 4, 8 and 12 weeks ahead, giving 12 dates in 2025.

At each origin, the model uses only observations available up to that week. Later origins can use earlier observed weeks from 2025, while future source values still come from our scenarios. This is a rolling-origin evaluation, not one year-long forecast.

We added the 0-100% output limit after inspecting the first 2025 run, so the results below are a revised evaluation. We use the per-period table and plot to see where the model works better or worse than the reference; we do not tune it further on this year.

Course: [ML approach](../../Curriculum/D2Dbook/4_Machine_Learning/2_Forecasting/2_ML_approach.ipynb), "Validation" and "Backtesting"; [SARIMAX](../../Curriculum/D2Dbook/4_Machine_Learning/2_Forecasting/3_SARIMAX.ipynb), "SARIMAX prediction".


In [68]:
# Freezing the validation-selected settings before scoring the reserved year.
test_settings = candidate_settings.copy()
test_origins = pd.date_range("2024-12-29", "2025-09-28", freq="13W-SUN")
test_rows, test_paths = [], []
for origin in test_origins:
    started = perf_counter()
    forecast, model = forecast_no3(no3_weekly, origin, **test_settings)
    elapsed = perf_counter() - started
    actual = no3_weekly.loc[forecast.index, "filling_percent"]
    reference = no3_weekly.loc[forecast.index - pd.Timedelta(weeks=52), "filling_percent"].to_numpy()
    test_rows.append({"origin": origin, "river_mae_pp": float((actual - forecast).abs().mean()),
                      "reference_mae_pp": float(np.abs(actual.to_numpy() - reference).mean()),
                      "seconds": elapsed})
    test_paths.append(pd.DataFrame({"origin": origin.strftime("%Y-%m-%d"),
        "Observed": actual, "River": forecast, "52-week reference": reference}).reset_index())

# Reporting per-origin errors and their equally weighted overall means, without retuning.
test_scores = pd.DataFrame(test_rows).set_index("origin")
test_paths = pd.concat(test_paths, ignore_index=True)
display(test_scores.round(3))
display(test_scores.mean().rename("Overall 2025").to_frame().T.round(3))
print("Predictions outside 0-100%:", int((~test_paths["River"].between(0, 100)).sum()))

# Showing each forecast origin separately, using the same three scored dates per panel.
import plotly.express as px

test_plot = test_paths.melt(id_vars=["origin", "date"], var_name="series", value_name="filling_percent")
test_figure = px.line(test_plot, x="date", y="filling_percent", color="series",
    facet_col="origin", facet_col_wrap=2, markers=True, color_discrete_map={"Observed": "black"},
    labels={"date": "Week ending", "filling_percent": "Filling (%)", "series": "Series"},
    title="NO3: reserved-2025 forecasts at 4, 8 and 12 weeks")
test_figure.update_layout(template="plotly_white", height=550, legend_title_text="Series")
test_figure


,river_mae_pp,reference_mae_pp,seconds
origin,,,
2024-12-29,6.158,32.419,0.027
2025-03-30,11.603,19.037,0.021
2025-06-29,6.794,9.706,0.020
2025-09-28,14.381,5.399,0.020


,river_mae_pp,reference_mae_pp,seconds
Overall 2025,9.734,16.64,0.022


Predictions outside 0-100%: 0


### Findings from 2025

We get an overall MAE of **9.73 pp**, compared with **16.64 pp** for the 52-week reference. River is better in three of four periods, but the reference is better in the final period (**14.38 versus 5.40 pp**). We observed overestimation in some periods.

We keep the tested setup as the candidate for the app and show the reference alongside it. The test covers only 12 dates and does not simulate publication delays, so the result should be read within those limits.


## 14. Forecast page and quantile bonus

The app opens with weekly predictions, two years of history, p=4, seasonal differencing and all sources. Four-week steps with p=1 gave the lowest validation MAE and are also selectable. We export only the target and supported inputs, keeping missing flow values unchanged. We used this CSV to seed ind320_ca4.no3_weekly in MongoDB. The app reads one document per NO3 week. We checked all 521 rows and reproduced the same 12 weekly forecast values after the transfer.

The first Update data click after Monday 06:00 in Oslo checks the sources. MongoDB records the completed check for all users, including when the newest week still has incomplete inputs. Later clicks that week return immediately. If a request fails, we can retry. The update checks published NO3 filling, exchange rates, the same 100 weather points and eight directional flow series. It checks the last two stored weeks again and appends complete weeks in order. Each document uses the region and Sunday date as its key, so refreshing a week updates the existing row. The forecast starts from the latest complete week unless we select an earlier period. For a missing ISO week 53 in the training weather, the app averages weeks 52 and 1.

On 27 November 2026, the first update added 46 complete weeks. The last observed filling level was 69.72% on 15 November. The week ending 22 November still had incomplete inputs. The default forecast used the 567 stored weeks as its available dataset, trained on the latest 104 and predicted through 7 February 2027.

The page is temporarily placed with the reservoir pages. We can choose training dates, horizon, individual inputs and model parameters. The period and horizon are stored in session state; connecting them to the map comes later. We also leave the overall design and structure sketch for that later step.



In [69]:
# Exporting the prepared observations used by the app, retaining the original missing flows.
import sys

app_folder = PROJECT_ROOT / "Compulsory Assignments"
if str(app_folder) not in sys.path:
    sys.path.insert(0, str(app_folder))
from modules.forecasting import EXOGENOUS_LABELS, forecast_quantiles
from modules.forecast_plotting import build_forecast_figure

forecast_data = no3_weekly[["filling_percent", *EXOGENOUS_LABELS]]
forecast_data.to_csv(app_folder / "data/ca4/no3_weekly.csv", index_label="date")
print(f"Exported {len(forecast_data)} weeks for the NO3 forecast page.")



Exported 521 weeks for the NO3 forecast page.


### Forecast intervals from past errors

We replay the chosen model at up to 20 earlier dates spaced four weeks apart. Each replay uses the selected history length and only outcomes known by the current forecast date. For each horizon we fit a constant to the absolute errors using River Quantile loss. The fitted 80th percentile gives the distance on either side of the red line, limited to 0-100% filling. We show a band when at least five comparable past forecasts are available.


In [ ]:
# Scoring both time resolutions at the same eight forecast origins.
rows = []
weekly_settings = dict(candidate_settings, step_weeks=1, p=4)
for settings in (weekly_settings, candidate_settings):
    for origin in origins:
        band = forecast_quantiles(no3_weekly, origin, coverage=0.8, **settings)
        actual = no3_weekly.loc[band.index, "filling_percent"]
        for covered, width in zip(actual.between(band.lower, band.upper), band.upper - band.lower):
            rows.append((settings["step_weeks"], covered, width))
scores = pd.DataFrame(rows, columns=["step_weeks", "covered", "width_pp"])
display(scores.groupby("step_weeks").agg(hits=("covered", "sum"), total=("covered", "size"),
                                         coverage=("covered", "mean"), width_pp=("width_pp", "mean")))

# Drawing the weekly forecast shown when the page opens.
latest_band = forecast_quantiles(no3_weekly, no3_weekly.index[-1], coverage=0.8, **weekly_settings)
build_forecast_figure(no3_weekly["filling_percent"], latest_band).show()


### What we see from the interval check

With the weekly default, the nominal 80% band covered **70 of 96 observations (73%)** and averaged **24.44 pp** wide. With four-week steps it covered **19 of 24 (79%)** and averaged **26.07 pp** wide. Both checks use eight forecast origins. The red line lies inside by construction, while coverage measures where the actual filling level landed.

The point model follows [Streaming models](../../Curriculum/D2Dbook/4_Machine_Learning/3_Streaming/2_Streaming_models.ipynb), under **Streaming forecasts** and **SARIMA + feature engineering**. The animation builds on [Plotly interactive](../../Curriculum/D2Dbook/5_Deployment2/2_Graphics/2_Plotly_interactive.ipynb), under **Animation**. The Quantile-loss extension is the selected assignment bonus; its loss is described in the [River documentation](https://riverml.xyz/latest/api/optim/losses/Quantile/).

